# Lesson 3.4 — AWS Backup, Snapshots & Data Versioning

**Level: Intermediate** | **Exam:** SOA-C03 Domain 2, Task 2.3

## Learning Objectives
- Automate snapshots for EC2 (EBS), RDS, DynamoDB, and S3 using AWS Backup
- Implement point-in-time restore for RDS and DynamoDB
- Enable S3 versioning and configure MFA Delete for data protection
- Calculate RTO/RPO implications of different backup strategies

_Reference: [AWS Backup](https://docs.aws.amazon.com/aws-backup/latest/devguide/) | [RDS PITR](https://docs.aws.amazon.com/AmazonRDS/latest/UserGuide/USER_PitrRestore.html)_

## Backup Options by Service

| Service | Backup Method | RPO | Retention Max |
|---------|-------------|-----|---------------|
| **EC2/EBS** | EBS Snapshots (incremental) | Snapshot frequency | Indefinite in S3 |
| **RDS** | Automated backups + manual snapshots | 5 min (PITR) | 35 days auto; indefinite manual |
| **DynamoDB** | On-demand backup + PITR | 5 min (PITR) | Indefinite backup; 35 days PITR |
| **S3** | Versioning + replication | Real-time | Indefinite |
| **EFS** | AWS Backup | Snapshot frequency | Policy-based |

## EBS Snapshots — Key Facts
- **Incremental**: Only changed blocks are stored after first full snapshot
- **Stored in S3** (but not visible in your S3 console — managed by AWS)
- Snapshots can be **cross-region copied** and **shared** across accounts
- **Data Lifecycle Manager (DLM)**: Policy-based automation of EBS snapshots

## RDS Point-in-Time Restore (PITR)
- Restores to **any second** within backup retention period (up to 35 days)
- Creates a **new DB instance** (doesn't overwrite existing)
- RPO = 5 minutes (transaction logs are backed up every 5 minutes)
- **Important**: You restore to a NEW endpoint — update connection strings!

## S3 Versioning
- Keeps **every version** of every object — protects against accidental deletion
- Deleting a versioned object creates a **delete marker** (recoverable)
- **MFA Delete**: Requires MFA to permanently delete versions or suspend versioning
- Use lifecycle policies to expire old versions and control costs

In [ ]:
import boto3, json

s3 = boto3.client('s3', region_name='us-east-1')
ec2 = boto3.client('ec2', region_name='us-east-1')
backup = boto3.client('backup', region_name='us-east-1')
account_id = boto3.client('sts').get_caller_identity()['Account']
print(f'Account: {account_id}')

In [ ]:
# Enable S3 versioning on a bucket
BUCKET = f'cloudops-versioning-demo-{account_id}'

try:
    s3.create_bucket(Bucket=BUCKET)
    print(f'Created bucket: {BUCKET}')
except Exception as e:
    print(f'Bucket: {e}')

# Enable versioning
s3.put_bucket_versioning(
    Bucket=BUCKET,
    VersioningConfiguration={'Status': 'Enabled'}
)
print('Versioning: ENABLED')

# Upload multiple versions of the same object
for version_num in range(1, 4):
    content = f'This is version {version_num} of the configuration file.'
    response = s3.put_object(
        Bucket=BUCKET,
        Key='config/app-settings.json',
        Body=content.encode()
    )
    print(f'Uploaded version {version_num}: VersionId={response["VersionId"][:16]}...')

# List all versions
versions = s3.list_object_versions(Bucket=BUCKET, Prefix='config/')['Versions']
print(f'\nAll versions of config/app-settings.json ({len(versions)}):') 
for v in versions:
    is_latest = 'LATEST' if v['IsLatest'] else '      '
    print(f'  {is_latest} {v["LastModified"].strftime("%H:%M:%S")} {v["VersionId"][:20]}...')

In [ ]:
# Restore a previous version from S3 versioning

# Get all versions
versions = s3.list_object_versions(Bucket=BUCKET, Prefix='config/')['Versions']
versions_sorted = sorted(versions, key=lambda x: x['LastModified'])

# Get the oldest version (version 1)
oldest_version = versions_sorted[0]
print(f'Restoring to oldest version: {oldest_version["VersionId"][:20]}...')

# Read the old version content
old_content = s3.get_object(
    Bucket=BUCKET,
    Key='config/app-settings.json',
    VersionId=oldest_version['VersionId']
)['Body'].read().decode()

print(f'Restored content: "{old_content}"')

# To restore: simply copy the old version to the same key
s3.copy_object(
    Bucket=BUCKET,
    Key='config/app-settings.json',
    CopySource={
        'Bucket': BUCKET,
        'Key': 'config/app-settings.json',
        'VersionId': oldest_version['VersionId']
    }
)
print('Successfully restored oldest version as current version.')

In [ ]:
# Amazon Data Lifecycle Manager (DLM) — automate EBS snapshots
dlm = boto3.client('dlm', region_name='us-east-1')

# Create an IAM role for DLM (in practice, use the pre-created AWSDataLifecycleManagerDefaultRole)
print('EBS Data Lifecycle Manager Policy Structure:')
dlm_policy = {
    'ResourceType': 'INSTANCE',  # Snapshot all EBS volumes attached to tagged instances
    'TargetTags': [{'Key': 'Backup', 'Value': 'true'}],
    'Schedules': [
        {
            'Name': 'Daily-Snapshots',
            'CreateRule': {
                'Interval': 24,
                'IntervalUnit': 'HOURS',
                'Times': ['04:00']          # 4 AM UTC daily
            },
            'RetainRule': {'Count': 7},     # Keep last 7 snapshots
            'CopyTags': True
        },
        {
            'Name': 'Weekly-CrossRegion',
            'CreateRule': {
                'Interval': 1,
                'IntervalUnit': 'WEEKS',
                'Times': ['02:00']
            },
            'RetainRule': {'Count': 4},
            'CrossRegionCopyRules': [
                {
                    'TargetRegion': 'us-west-2',
                    'Encrypted': True,
                    'RetainRule': {'Interval': 30, 'IntervalUnit': 'DAYS'}
                }
            ]
        }
    ],
    'PolicyType': 'EBS_SNAPSHOT_MANAGEMENT'
}

import json
print(json.dumps(dlm_policy, indent=2))
print('\nTo apply: dlm.create_lifecycle_policy(PolicyDetails=dlm_policy, ExecutionRoleArn=role_arn, ...)')

## Exercise

In [ ]:
# TODO: Implement the S3 versioning lifecycle — add a lifecycle policy to the demo bucket that:
# 1. Expires non-current versions after 30 days
# 2. Removes delete markers that have no non-current versions (expired object delete markers)
# This controls costs from versioning by not retaining old versions forever

import boto3
s3 = boto3.client('s3', region_name='us-east-1')
account_id = boto3.client('sts').get_caller_identity()['Account']
BUCKET = f'cloudops-versioning-demo-{account_id}'

lifecycle_policy = {
    'Rules': [
        {
            'ID': 'ManageVersionedObjects',
            'Status': 'Enabled',
            'Filter': {'Prefix': ''},  # Apply to all objects
            # TODO: Add NoncurrentVersionExpiration
            # TODO: Add ExpiredObjectDeleteMarker
        }
    ]
}

# YOUR CODE HERE to apply the lifecycle policy

In [ ]:
# Cleanup
import boto3
s3 = boto3.client('s3', region_name='us-east-1')
account_id = boto3.client('sts').get_caller_identity()['Account']
BUCKET = f'cloudops-versioning-demo-{account_id}'

try:
    # Delete all object versions
    versions = s3.list_object_versions(Bucket=BUCKET).get('Versions', [])
    delete_markers = s3.list_object_versions(Bucket=BUCKET).get('DeleteMarkers', [])
    objects_to_delete = [{'Key': v['Key'], 'VersionId': v['VersionId']} for v in versions + delete_markers]
    if objects_to_delete:
        s3.delete_objects(Bucket=BUCKET, Delete={'Objects': objects_to_delete})
    s3.delete_bucket(Bucket=BUCKET)
    print(f'Deleted bucket: {BUCKET}')
except Exception as e:
    print(f'Cleanup: {e}')

## Summary
- **EBS Snapshots**: Incremental, stored in S3, support cross-region copy. Automate with DLM.
- **RDS PITR**: Restores to any second in the retention window (1–35 days). Creates a NEW DB instance.
- **S3 Versioning**: Protects against accidental deletion and overwrites. Add lifecycle rules to manage costs.
- **AWS Backup**: Central service to manage backup policies across EC2, RDS, EFS, DynamoDB, and S3.

---
**Next Phase:** [Phase 4 — Deployment, Provisioning & Automation](../phase-4-deployment/lesson-4.1-amis-image-builder.ipynb)